In [1]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/scoring"
    !pip install -q interpret shap fairlearn lightgbm xgboost
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.insert(0, PROJECT_ROOT)
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

# Phase 2: Black-Box Benchmarks + SHAP

## 2.1 Train baselines (LR, XGBoost, EBM) on all features

In [2]:
import pandas as pd
from pathlib import Path

PROCESSED = Path(PROJECT_ROOT) / "data" / "processed"
X_train = pd.read_csv(PROCESSED / "X_train_raw.csv")
X_test  = pd.read_csv(PROCESSED / "X_test_raw.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED / "y_test.csv").squeeze()
train_dates = pd.to_datetime(X_train["issue_d"])
test_dates  = pd.to_datetime(X_test["issue_d"])
print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (684592, 61) Test: (1096128, 61)


In [3]:
from src.config import CONFIG
from src.models import MODEL_REGISTRY
from src.validation import make_time_series_splits, cross_validate_model

splits = make_time_series_splits(
    train_dates,
    n_splits=CONFIG.n_splits_outer,
    embargo_months=CONFIG.embargo_months,
    min_train_months=CONFIG.min_train_months,
)
print(f"Built {len(splits)} time-series folds on training portion")

cv_summaries = {}
for name in ("dummy", "lr", "xgb", "lgbm", "ebm"):
    builder = MODEL_REGISTRY[name]
    print(f"\n== Running reporting CV for {name} ==")
    df = cross_validate_model(
        pipeline_factory=lambda b=builder: b(y_train=y_train, params=None),
        X_raw=X_train, y=y_train, dates=train_dates, splits=splits,
        model_name=f"{name}_baseline",
        reports_dir=Path(PROJECT_ROOT) / "reports",
    )
    cv_summaries[name] = df
    print(df.tail(1))


Built 5 time-series folds on training portion

== Running reporting CV for lr ==


/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['acc_open_pa

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with str

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with str

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with str

...

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['acc_open_pa

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['acc_open_pa

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['acc_open_pa

/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['tot_hi_cred_lim' 'avg_cur_bal' 'total_bc_limit' 'mort_acc'
 'bc_open_to_buy' 'bc_util' 'mo_sin_old_il_acct' 'mo_sin_old_rev_tl_op'
 'mo_sin_rcnt_rev_tl_op' 'mo_sin_rcnt_tl' 'mths_since_recent_bc'
 'mths_since_recent_inq' 'num_accts_ever_120_pd' 'num_actv_bc_tl'
 'num_actv_rev_tl' 'num_bc_sats' 'num_bc_tl' 'num_il_tl' 'num_op_rev_tl'
 'num_rev_accts' 'num_rev_tl_bal_gt_0' 'num_sats' 'num_tl_30dpd'
 'num_tl_90g_dpd_24m' 'num_tl_op_past_12m' 'pct_tl_nvr_dlq'
 'percent_bc_gt_75' 'tot_coll_amt' 'tot_cur_bal' 'total_bal_ex_mort'
 'total_il_high_credit_limit' 'total_rev_hi_lim']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['acc_open_pa

       fold            AUPRC            AUROC               F1  \
5  mean±std  0.2553 ± 0.0624  0.6618 ± 0.0327  0.0534 ± 0.0546   

  balanced_accuracy  
5   0.5105 ± 0.0110  


In [4]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path(PROJECT_ROOT) / "reports" / "cv_results"
out.mkdir(parents=True, exist_ok=True)

for name, df in cv_summaries.items():
    fold_rows = df[df["fold"] != "mean±std"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(fold_rows["fold"].astype(str), fold_rows["AUPRC"], color="#4c72b0")
    ax.axhline(fold_rows["AUPRC"].mean(), color="k", linestyle="--", label="mean")
    ax.errorbar([len(fold_rows) / 2 - 0.5], [fold_rows["AUPRC"].mean()],
                yerr=[fold_rows["AUPRC"].std()], fmt="none", ecolor="k", capsize=8)
    ax.set_title(f"{name} baseline (75 feat) — AUPRC per fold")
    ax.set_xlabel("fold"); ax.set_ylabel("AUPRC"); ax.legend()
    fig.tight_layout()
    fig.savefig(out / f"{name}_baseline_folds.png", dpi=150)
    plt.close(fig)
print("Fold bar charts saved to reports/cv_results/")


Fold bar charts saved to reports/cv_results/


## 2.4 Final fit on full train + SHAP feature importance

In [5]:
import joblib
from pathlib import Path

MODELS = Path(PROJECT_ROOT) / "models"
MODELS.mkdir(parents=True, exist_ok=True)

final_models = {}
for name in ("dummy", "lr", "xgb", "lgbm", "ebm"):
    print(f"Fitting final {name} on full train...")
    pipe = MODEL_REGISTRY[name](y_train=y_train, params=None)
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, MODELS / f"{name}.pkl")
    final_models[name] = pipe
print("Saved baseline (75-feature) Pipelines to models/")


Fitting final lr on full train...


/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=8', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Fitting final xgb on full train...


Fitting final lgbm on full train...


[LightGBM] [Warning] seed is set=42, random_state=42 will be ignored. Current value: seed=42


[LightGBM] [Warning] seed is set=42, random_state=42 will be ignored. Current value: seed=42
[LightGBM] [Info] Number of positive: 120246, number of negative: 564346
[LightGBM] [Info] Total Bins 6622
[LightGBM] [Info] Number of data points in the train set: 684592, number of used features: 131
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.175646 -> initscore=-1.546128
[LightGBM] [Info] Start training from score -1.546128


Fitting final ebm on full train...


/Users/abdro/anaconda3/envs/scoring/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Saved baseline (75-feature) Pipelines to models/


In [ ]:
from src.evaluate import bootstrap_metrics, metrics_table
from pathlib import Path

test_results = {}
for name, pipe in final_models.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    test_results[name] = bootstrap_metrics(
        y_test.to_numpy(), y_pred, y_prob, n_bootstraps=1000, seed=42,
    )

tbl = metrics_table(test_results)
out = Path(PROJECT_ROOT) / "reports" / "test_metrics_baseline_ci.csv"
out.parent.mkdir(parents=True, exist_ok=True)
tbl.to_csv(out)
print(tbl)

In [6]:
import shap
import numpy as np
import pandas as pd
from pathlib import Path

REPORTS = Path(PROJECT_ROOT) / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

xgb_pipe = final_models["xgb"]
X_test_pre = xgb_pipe.named_steps["pre"].transform(X_test)

# Pull feature names from the ColumnTransformer
pre = xgb_pipe.named_steps["pre"]
try:
    feature_names = pre.get_feature_names_out()
except Exception:
    feature_names = None

# Subsample for SHAP speed (SHAP on 1M+ rows is impractical)
rng = np.random.default_rng(42)
n_shap = min(20000, len(X_test_pre))
idx = rng.choice(len(X_test_pre), size=n_shap, replace=False)
X_shap = X_test_pre[idx]

explainer = shap.TreeExplainer(xgb_pipe.named_steps["clf"])
shap_values = explainer.shap_values(X_shap)

# Summary plot
plt.figure()
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False, max_display=20)
plt.tight_layout()
plt.savefig(REPORTS / "shap_summary_baseline.png", dpi=150, bbox_inches="tight")
plt.close()

# SHAP importance (mean |SHAP|) for top-10 selection
abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_names if feature_names is not None else [f"f{i}" for i in range(len(abs_shap))],
    "importance": abs_shap,
}).sort_values("importance", ascending=False).reset_index(drop=True)
importance_df.to_csv(REPORTS / "shap_importance_baseline.csv", index=False)
print("SHAP summary + importance saved.")
print(importance_df.head(15))


SHAP summary + importance saved.
   feature  importance
0       f3    0.345288
1     f113    0.183211
2       f4    0.135182
3       f1    0.125548
4      f44    0.123512
5       f2    0.117481
6       f0    0.075558
7      f71    0.066732
8      f70    0.064103
9       f7    0.060380
10     f38    0.053162
11     f34    0.052533
12     f15    0.051793
13     f18    0.048147
14     f13    0.045116
